In [ ]:
#!/usr/bin/env python3

import json
from pathlib import Path
from typing import Tuple

from plotly.io import renderers

import hail as hl
from hail.ggplot import *  # noqa: F403

renderers.default = 'notebook_connected'

In [ ]:
def import_table(path: Path) -> hl.Table:
    runt = hl.tstruct(
        iteration=hl.tint,
        failure=hl.tstr,
        timed_out=hl.tbool,
        time=hl.tfloat,
        task_memory=hl.tfloat,
    )

    with path.open(encoding='utf-8') as fp:
        rows = [json.loads(line) for line in fp]

    ht = hl.Table.parallelize(rows, partial_type={'runs': hl.tarray(runt)})
    ht = ht.annotate(trial=hl.parse_int(ht.trial))
    ht = ht.rename({'trial': 'instance', 'runs': 'trials'})
    return ht.key_by(ht.path, ht.name, ht.version, ht.instance)


def filter_failures_and_outliers(ht: hl.Table) -> hl.Table:
    return ht.annotate(
        runs=hl.rbind(
            ht.trials.filter(lambda t: hl.is_missing(t.failure) | ~t.timed_out),
            lambda trials: hl.rbind(
                hl.median(trials.map(lambda r: r.time)),
                lambda median: trials.filter(lambda r: (r.time / median) < 10),
            ),
        )
    )

ht = import_table(Path('../results.jsonl'))
ht = filter_failures_and_outliers(ht)
ht = ht.checkpoint(hl.utils.new_local_temp_file())
benchmarks = ht.key_by(ht.path, ht.name).distinct().name.collect()
print(*benchmarks, sep='\n')

In [ ]:
for name in benchmarks:
    k = ht.filter(ht.name == name)
    k = k.explode(k.trials, name='trial')
    (
        ggplot(k, aes(x=k.trial.iteration, y=k.trial.time, color=hl.str(k.instance))) +
        geom_line() +
        ggtitle(name) #
    ).show()

In [ ]:
def filter_burn_in_iterations(ht: hl.Table) -> hl.Table:
    ht = ht.annotate_globals(
        first_stable_index={
            'benchmark_export_range_matrix_table_row_p100': 15,
            'benchmark_import_gvcf_force_count': 10,
            'benchmark_matrix_table_take_col': 15,
            'benchmark_ndarray_matmul_int64': 15,
            'benchmark_sample_qc': 11,
            'benchmark_shuffle_key_rows_by_mt': 8,
            'benchmark_union_partitions_table[100-100]': 20,
        },
    )

    return ht.annotate(
        trials=(
            hl.enumerate(ht.trials)
            .filter(lambda t: t[0] >= ht.first_stable_index[ht.name])
            .map(lambda t: t[1]) # ruff
        ),
    )

ht = filter_burn_in_iterations(ht)
ht = ht.checkpoint(hl.utils.new_local_temp_file())

In [ ]:
for name in benchmarks:
    k = ht.filter(ht.name == name)
    k = k.explode(k.trials, name='trial')
    (
        ggplot(k, aes(x=k.trial.iteration, y=k.trial.time, color=hl.str(k.instance))) +
        geom_line() +
        ggtitle(name) # ruff
    ).show()

In [ ]:
# laaber et al. section 4

def cv(run: hl.StructExpression) -> hl.Float64Expression:
    return hl.bind(lambda s: s.stdev / s.mean, hl.agg.stats(run.time))


def variability(ht: hl.Table) -> hl.Table:
    key_wo_instance = ht.key.drop('instance')
    trials = ht.annotate(cv=ht.trials.aggregate(cv))

    trials = (
        trials
        .group_by(*key_wo_instance)
        .aggregate(stats=hl.agg.stats(trials.cv).select('mean', 'stdev')) #
    )

    variability = (
        ht
        .group_by(*key_wo_instance)
        .aggregate(total=hl.agg.explode(cv, ht.runs)) #
    )

    return variability.annotate(trials=trials[variability.key].stats)


variability(ht).show()

In [ ]:
for name in benchmarks:
    k = ht.filter(ht.name == name)
    k = k.annotate(s=k.trials.aggregate(lambda r: hl.agg.stats(r.time)))
    (ggplot(k, aes(x=k.instance, y=k.s.mean)) + geom_point() + ggtitle(name)).show()

In [ ]:
# laaber et al. section 5


def randomize_with_replacement(xs: hl.ArrayExpression) -> hl.ArrayExpression:
    return hl.bind(
        lambda xs_: hl.map(
            lambda idx: xs_[idx],
            hl.bind(lambda len: hl.repeat(lambda: hl.rand_int32(len), len), hl.len(xs_))
        ),
        xs,
    )


def select_disjoint(xs: hl.ArrayExpression, pos: hl.Int32Expression) -> hl.TupleExpression:
    return hl.bind(
        lambda xs_: hl.bind(
            lambda len: hl.case()
            .when(pos <= 2 * len, hl.bind(lambda ys: hl.tuple([ys[:pos], ys[pos:2*pos]]), hl.shuffle(xs_)))
            .or_error("split position '" + hl.str(pos) + "' exceeds twice array length '" + hl.str(len) + "'."),
            hl.len(xs_),
        ),
        xs,
    )


def run_bootstrap_simulations(
    ht: hl.Table,
    samples: int,
    confidence: float,
) -> hl.Table:
    if confidence <= 0 or confidence >= 1:
        raise ValueError(f'Confidence must fall within interval (0, 1), got {confidence}.')

    sim = (
        ht.group_by(*ht.key.drop('instance'))
        .aggregate(instances=hl.agg.collect(ht.instance))
        .annotate(samples=hl.range(samples))
        .explode('samples', name='sample')
    )

    sim = (
        sim.annotate(instances=hl.sorted(randomize_with_replacement(sim.instances)))
        .explode('instances', name='instance')
        .key_by(*ht.key)  #
    )

    sim = sim.annotate(
        trials=hl.map(
            lambda t: t.time,
            randomize_with_replacement(ht[sim.key].trials),
        ),
    )

    sim = (
        sim
        .group_by(*sim.key.drop('instance'), 'sample')
        .aggregate(mean=hl.agg.explode(hl.agg.mean, sim.trials))  #
    )

    sim = (
        sim
        .group_by(*sim.key.drop('sample'))
        .aggregate(means=hl.agg.collect(sim.mean))  #
    )

    endpoints = (lower := (1 - confidence) / 2, confidence - lower)

    return sim.select(
        **hl.bind(
            lambda means, len: hl.bind(
                lambda lo, hi: hl.struct(
                    ci=hl.interval(lo, hi, includes_end=True),
                    radius=hl.rbind((hi + lo) / 2, lambda mid: (hi - mid) / mid),
                ),
                *[means[hl.int(p * len)] for p in endpoints],
            ),
            hl.sorted(sim.means),
            hl.len(sim.means),
        ),
    )


def ibs(
    ht: hl.Table,
    ninstances: hl.Int32Expression,
    ntrials: hl.Int32Expression,
) -> Tuple[hl.Table, hl.Table]:
    def normalize(ht: hl.Table, key: hl.StructExpression) -> hl.Table:
        ht = ht.explode('instances', name='_s')
        return ht.select(**ht._s)._key_by_assert_sorted(*key)

    sel = (
        ht
        .group_by(*ht.key.drop('instance'))
        .aggregate(
            **hl.bind(
                lambda instances: hl.bind(
                    lambda *groups: hl.struct(**dict(zip(
                        ('control', 'test'),
                        (hl.sorted(g, key=lambda i: i.instance) for g in groups)
                    ))),
                    *select_disjoint(instances, ninstances),
                ),
                hl.agg.collect(
                    hl.struct(
                        instance=ht.instance,
                        trials=hl.shuffle(ht.trials)[:ntrials],
                    )
                ),
            )
        )
    )

    sel = sel.checkpoint(hl.utils.new_local_temp_file())
    control = normalize(sel.select(instances=sel.control), ht.key)
    test = normalize(sel.select(instances=sel.test), ht.key)
    return (control, test)


def tbs(
    ht: hl.Table,
    ninstances: hl.Int32Expression,
    ntrials: hl.Int32Expression,
) -> Tuple[hl.Table, hl.Table]:
    sel = (
        ht.group_by(*ht.key.drop('instance'))
        .aggregate(
            instances=hl.map(
                lambda instance: instance.select(
                    instance=instance.instance,
                    **dict(zip(('control', 'test'), select_disjoint(instance.trials, ntrials))),
                ),
                hl.sorted(
                    hl.shuffle(hl.agg.collect(ht.row))[:ninstances],
                    key=lambda i: i.instance,
                ),
            ),
        )
        .explode('instances', name='_s')
    )

    sel = (
        sel
        .select(**sel._s)
        ._key_by_assert_sorted(*ht.key)
        .checkpoint(hl.utils.new_local_temp_file())
    )

    return (sel.select(trials=sel.control), sel.select(trials=sel.test))


def confidence_interval_test(
    control: hl.Table,
    test: hl.Table,
    samples: hl.Int32Expression,
    confidence: hl.Float64Expression,
) -> hl.Table:
    control = run_bootstrap_simulations(control, samples, confidence)
    test = run_bootstrap_simulations(test, samples, confidence)
    return control.select(overlaps=test[control.key].ci.overlaps(control.ci))


def scale(ht: hl.Table, factor: hl.Float64Expression) -> hl.Table:
    return ht.annotate(trials=ht.trials.map(lambda t: t.annotate(time=t.time * factor)))


ibc, ibt = ibs(ht, ninstances=5, ntrials=5)
tbc, tbt = tbs(ht, ninstances=5, ntrials=5)

for t in (ibc, ibt, tbc, tbt):
    t.show()

In [ ]:
# A/A Testing
print('Instance Based Sampling:')
confidence_interval_test(ibc, ibt, 1000, 0.95).show()
print('Trial Based Sampling:')
confidence_interval_test(tbc, tbt, 1000, 0.95).show()

In [ ]:
# Instance-Based-Sampling A/B Testing with Simulated Slowdown
factor = 1.2

ab = (
    ht
    .annotate(
        ninstances=[1, 2, 3, 5, 10, 20],
        ntrials=[1, 2, 3, 5],
        experiment=hl.range(100),
    )
    .explode('ninstances')
    .explode('ntrials')
    .explode('experiment')
)

ab = ab.key_by(*ab.key.drop('instance'), 'ninstances', 'ntrials', 'experiment', 'instance')

control, test = tbs(ab, hl.agg.take(ab.ninstances, 1)[0], hl.agg.take(ab.ntrials, 1)[0])
results = confidence_interval_test(control, scale(test, factor), 1000, 0.95)
results = (
    results
    .group_by(*results.key.drop('experiment'))
    .aggregate(rate=hl.agg.count_where(~results.overlaps) / hl.agg.count())
)

results.filter(results.rate>=0.95).show()